# 3 Where does a good representation come from?

In `01_convolutions.ipynb` you built a CNN by hand and trained it on fashion MNIST. Two of the
three possible answers to that question are now behind you:

1. **you design it** — kernel size, stride, padding, how many layers, where to pool. Those were
   your choices, and you made them by reasoning about output shapes.
2. **you learn it** — the filter weights themselves, fitted to fashion MNIST by gradient descent.

This notebook is the third answer, and it is the one most of the field actually reaches for:
**you take a representation that someone else already learned.** A vision model trained on a
million-plus photographs has filters for edges, textures and object parts already. You did not pay
for them, and you do not have to retrain them.

The move is exactly the one from `1_pytorch_intro/07_vectors.ipynb`: run a frozen encoder over the
data **once**, cache the vectors, then train a small head on top. There it was movie reviews and a
regex baseline. Here it is images and the CNN you just built. **The `Collate` and the head are
literally the same code** — which is the point we come back to at the end.

In [ ]:
import time
import warnings
from pathlib import Path

import numpy as np
import torch
import torch.optim as optim
from datasets import load_dataset
from torch.utils.data import DataLoader

warnings.simplefilter("ignore", UserWarning)

ARTEFACTS = Path("../../artefacts").resolve()
ARTEFACTS.mkdir(exist_ok=True)

## 3.1 The frozen encoder

`ImageVectorizer` wraps any HuggingFace vision model: image in, one vector out. We default to
`mobilenet_v2`, a 3.5M-parameter CNN — small enough that the one-time pass below finishes while
you read this section.

In [ ]:
from vectormesh import ImageVectorizer

vectorizer = ImageVectorizer(model_name="google/mobilenet_v2_1.0_224")
vectorizer.get_hidden_size

### That "LOAD REPORT" is not an error

Loading the model prints a report with a couple of `UNEXPECTED` keys:

```
classifier.bias   | UNEXPECTED |
classifier.weight | UNEXPECTED |
```

Read it carefully, because it says exactly what we want to hear. The checkpoint is a
*classification* model: a **backbone** (the stack of convolutions) plus a **`classifier`** head
mapping features to 1000 ImageNet classes. `ImageVectorizer` loads it with `AutoModel`, which
builds the backbone **without any task head** — so the classifier weights in the file have nowhere
to go, and are reported as "these were in the file but the model did not use them".

That is the whole trick. We keep the backbone and throw the ImageNet head away, because we want
the *representation*, not ImageNet's 1000 labels.

- `UNEXPECTED` → benign: weights in the file we do not need.
- `MISSING` → the worrying one: backbone weights *not* found, leaving parts randomly initialised.

### One image, one vector

Before anything else, look at what the encoder does to a single image.

In [ ]:
flowers_peek = load_dataset("nkirschi/oxford-flowers", split="test")
image = flowers_peek[0]["image"]
print(f"input: {image.size} pixels, mode {image.mode}")

out = vectorizer([image], batchsize=1)
out["embed"][0].shape

`(500, 523)` colour pixels became `(1280,)` numbers.

On the ladder ([tensor-ladder.md](../1_pytorch_intro/tensor-ladder.md)) that is a move from
`(C, H, W)` down to `(D)` — **the encoder is the thing that destroys the spatial structure**,
after using it. Your CNN did the same thing internally: `convolutions` → `AvgPool2d` → `Flatten`
gave you `(B, filters)` before the dense layers. The difference is only *who trained the part in
front of the flatten*, and on how much data.

Note also what did **not** happen: no chunk axis. A review in `07_vectors.ipynb` was too long for
the encoder's context window and became `(C, D)`, needing a padder and an aggregator. An image
fits in one forward pass, so it is one vector, and the pipeline below is one rung shorter.

## 3.2 Two datasets, on purpose

**Fashion MNIST** is the *control*. It is the same task you just trained the CNN on, so the
comparison is honest — same data, same classes, same split.

**Oxford flowers** is the *headline*: 102 fine-grained classes of natural photographs. It is the
kind of problem where a small CNN trained from scratch struggles and a pretrained encoder should
shine. Before you run anything, predict which of the two gaps will be bigger, and why.

We subsample both to keep the one-time encoding pass to a few minutes on a CPU. Two details worth
copying:

- **`.shuffle()` before `.select()`.** The flowers train split is sorted by label, so
  `select(range(4000))` without a shuffle would silently hand you ~57 of the 102 classes and a
  model that looks mysteriously bad.
- **`DatasetSchema.infer`** finds the image and label columns, so switching datasets is one string
  and nothing downstream needs to know whether the column is called `image` or `img`.

Note what we do *not* do here: any `.map()` over the raw dataset. A `map` on a table with an image
column rewrites every image to disk, which on 4000 photographs costs far more than the encoding
does. One-hot encoding waits until §3.5, when the images are gone and a row is just a vector and a
label. **Do cheap operations on small data**, which after caching is everything.

In [ ]:
from vectormesh.data import DatasetSchema

N_TRAIN, N_TEST = 4000, 1000


def prepare(hub_id: str, n_train: int = N_TRAIN, n_test: int = N_TEST):
    ds = load_dataset(hub_id)
    schema = DatasetSchema.infer(ds["train"])
    labels = ds["train"].features[schema.label_col].names

    # shuffle FIRST: flowers is sorted by label, so a plain select() truncates the classes
    train = ds["train"].shuffle(seed=42).select(range(min(n_train, len(ds["train"]))))
    test = ds["test"].shuffle(seed=42).select(range(min(n_test, len(ds["test"]))))
    return train, test, schema, len(labels)


datasets = {
    "fashion": {"hub": "zalando-datasets/fashion_mnist", "size": (28, 28), "channels": 1},
    "flowers": {"hub": "nkirschi/oxford-flowers", "size": (64, 64), "channels": 3},
}

for name, cfg in datasets.items():
    train, test, schema, num_classes = prepare(cfg["hub"])
    cfg.update(train=train, test=test, schema=schema, num_classes=num_classes)
    print(f"{name:8} {num_classes:3} classes | {len(train)} train | {len(test)} test | {schema}")

## 3.3 What does encoding cost?

Before running the encoder over everything, measure it on a handful of images. This is a habit
worth having: the *rate* is what lets you predict the cost of a job before you start it, and it is
the number that ends up in the trade-off at the end of this notebook.

In [ ]:
sample = [flowers_peek[i]["image"] for i in range(64)]
vectorizer(sample[:8], batchsize=8)  # warm up: the first call pays for setup

start = time.time()
vectorizer(sample, batchsize=32)
rate = len(sample) / (time.time() - start)
print(f"{rate:.0f} images/second on {vectorizer.device}")

Hold onto that number. At this rate, encoding a dataset of any size is simple arithmetic — and
because it happens **once**, it is a cost you can amortise over every experiment you will ever run
on those vectors.

## 3.4 Building the cache

`VectorCache.create` runs the encoder over the dataset and writes the vectors to disk.
`remove_columns=[schema.input_col]` drops the raw pixels, so what lands in `artefacts/` is a small
folder of vectors and labels — cheap to copy, cheap to publish, and it is what somebody else can
download instead of buying a GPU.

This is the slow cell of the notebook. Everything after it is fast, and that asymmetry *is* the
lesson: you are paying once, here, for something you will reuse in every experiment below.

> Two caveats on the wall-clock you are about to see. First, `VectorCache.create` currently carries
> a fixed overhead of roughly 20 seconds per call on top of the encoding itself, so at the small
> sample sizes used here the reported time is mostly *not* encoding — trust the rate you just
> measured for the per-image cost. Second, we deliberately subsample; the honest full-dataset
> figure is the extrapolation printed at the end.

In [ ]:
from huggingface_hub import snapshot_download

from vectormesh import VectorCache

CACHE_REPO = "pttrn-io/mads-image-caches"  # org exists, cache not published yet — masterplan P0.3


def get_cache(tag: str, dataset, schema, column: str = "embed") -> tuple[VectorCache, float]:
    """Load a cache if we can, build it if we must. Returns (cache, seconds spent building)."""
    local = sorted(ARTEFACTS.glob(f"*_{tag}_{column}"))
    if local:
        print(f"loading {local[-1].name} from disk")
        return VectorCache.load(local[-1]), 0.0

    try:
        path = snapshot_download(
            repo_id=CACHE_REPO,
            repo_type="dataset",
            allow_patterns=f"{tag}/*",
            local_dir=ARTEFACTS / "hub",
        )
        print(f"downloaded {tag} from {CACHE_REPO}")
        return VectorCache.load(Path(path) / tag), 0.0
    except Exception as e:
        print(f"no published cache ({type(e).__name__}); building {tag} locally")
        start = time.time()
        cache = VectorCache.create(
            cache_dir=ARTEFACTS,
            vectorizer=vectorizer,
            dataset=dataset,
            dataset_tag=tag,
            remove_columns=[schema.input_col],
        )
        return cache, time.time() - start


for name, cfg in datasets.items():
    traincache, t1 = get_cache(f"{name}_train", cfg["train"], cfg["schema"])
    testcache, t2 = get_cache(f"{name}_test", cfg["test"], cfg["schema"])
    cfg.update(traincache=traincache, testcache=testcache, embed_time=t1 + t2)
    print(f"{name}: encoding took {t1 + t2:.0f}s for {len(traincache) + len(testcache)} images")

### Read the metadata before you write model code

Same habit as `07_vectors.ipynb`: the `metadata.json` is the public API of a cache, and it tells
you the architecture you need before you import anything.

In [ ]:
datasets["flowers"]["traincache"].metadata["embed"]

Two fields decide the whole pipeline:

- **`hidden_size: 1280`** — that is `D`, so the head's input size is 1280.
- **`tensordtype: 1`** — one item is a rank-1 `(D)` tensor. Every item therefore has the *same*
  shape, so batching is a plain `torch.stack`: **no padder, and no aggregator.**

Compare that with the granite text cache in `07_vectors.ipynb`, where `tensordtype: 2` meant
`(C, D)` items of varying length, which forced a `FixedPadding` and a `MaskedMeanAggregator` into
the pipeline. Same file format, same reading habit, different answer — and you knew the answer
before running anything.

## 3.5 The head

Now that the images are gone, one-hot encoding the label is cheap — `OneHot` maps over the *cache*,
not over the pictures. `Collate` then turns cached rows into batches, `Serial` chains components,
and the only thing between the cache and the loss is a two-layer `NeuralNet` plus one piece of
augmentation we will explain in a moment.

In [ ]:
from vectormesh.components import GaussianNoise, NeuralNet, Serial
from vectormesh.data import Collate, OneHot

collate_fn = Collate(embedding_col="embed", target_col="onehot", padder=torch.stack)


def cached_loaders(cfg, batch_size: int = 32):
    onehot = OneHot(
        num_classes=cfg["num_classes"], label_col=cfg["schema"].label_col, target_col="onehot"
    )
    return (
        DataLoader(cfg["traincache"].map(onehot), batch_size=batch_size, shuffle=True,
                   collate_fn=collate_fn),
        DataLoader(cfg["testcache"].map(onehot), batch_size=batch_size, shuffle=False,
                   collate_fn=collate_fn),
    )


def make_head(cfg, sigma: float = 0.1):
    hidden_size = cfg["traincache"].metadata["embed"]["hidden_size"]
    return Serial([
        GaussianNoise(sigma=sigma),                                   # (B, D) -> (B, D)
        NeuralNet(hidden_size=hidden_size, out_size=cfg["num_classes"]),  # (B, D) -> (B, K)
    ])


trainloader, testloader = cached_loaders(datasets["flowers"])
X, y = next(iter(trainloader))
head = make_head(datasets["flowers"])
print("X:", tuple(X.shape), "| y:", tuple(y.shape), "| logits:", tuple(head(X).shape))

`(B, D) -> (B, K)`. That is the entire model: two `Linear` layers over a frozen vector.

### Augmenting where the data now lives

Pixel augmentation — random crops and flips — wants a *fresh* view every epoch. That fights the
cache head on: a new crop means a new image, which means running the frozen encoder again, which
is the expensive thing we just paid for once.

So we augment in **feature space** instead. `GaussianNoise` perturbs the cached vector itself:
free, fresh on every step, and a decent regulariser for a small head. It is active in `train()`
mode and a no-op in `eval()`, so validation stays deterministic. Check that, do not trust it:

In [ ]:
head.train()
noisy_a, noisy_b = head(X), head(X)
head.eval()
clean_a, clean_b = head(X), head(X)

print("train mode varies between calls:", not torch.allclose(noisy_a, noisy_b))
print("eval mode is deterministic:     ", torch.allclose(clean_a, clean_b))

It is worth being honest about what this does and does not buy. Cropping a flower teaches the
model that a flower is still a flower when it moves; Gaussian noise in feature space only teaches
it not to depend too hard on any single one of 1280 coordinates. It is a weaker, cheaper
invariance. Exercise E2 asks you to find where it stops helping.

## 3.6 Training both models on the same data

One `train` function, used for the head and for the CNN, so the comparison cannot be rigged by an
accidental difference in the training setup. Both get the same optimizer, the same loss, the same
number of epochs and the same metric.

The CNN is the one **you** built in `01_convolutions.ipynb`, unchanged. That matters for the
comparison to mean anything: the thing being measured against a pretrained encoder should be the
thing you actually understand, not a black box from a library.

It lives in `cnn.py` next to this notebook rather than being pasted in here, because
`03_mlflow.ipynb` runs the same comparison and the two must not drift apart. A model copied into
two notebooks is a model that will be edited in one of them.

It also collects a debt from the last notebook. `_conv_test` runs a dummy tensor through the
convolutions to discover the activation-map size, so `AvgPool2d` always reduces it to
`(B, filters, 1, 1)`. That is what lets the *same class* take 28x28 greyscale fashion images and
64x64 colour flowers without a single change. Had we flattened instead, the first `Linear` would
need a different input size per dataset and we would be editing the model between runs.

In [ ]:
from cnn import CNN  # the model from 01_convolutions.ipynb

CNN??

In [ ]:
from mltrainer import ReportTypes, Trainer, TrainerSettings

from vectormesh.components.metrics import Accuracy
from vectormesh.data.vectorizers import detect_device

EPOCHS = 10
device = detect_device()
print(f"training on {device}")


def train(model, trainloader, testloader, epochs: int = EPOCHS, device: str = device):
    """Train, then return (accuracy, seconds)."""
    settings = TrainerSettings(
        epochs=epochs,
        metrics=[Accuracy()],
        logdir=Path("modellog").resolve(),
        train_steps=len(trainloader),
        valid_steps=len(testloader),
        reporttypes=[ReportTypes.TENSORBOARD],
        earlystop_kwargs=None,
    )
    trainer = Trainer(
        model=model,
        settings=settings,
        loss_fn=torch.nn.CrossEntropyLoss(),
        optimizer=optim.Adam,
        traindataloader=trainloader,
        validdataloader=testloader,
        scheduler=optim.lr_scheduler.ReduceLROnPlateau,
        device=device,
    )
    start = time.time()
    trainer.loop()
    elapsed = time.time() - start

    # loop() logs the metrics but does not keep them; one more eval pass to read the final number
    metric_dict, _ = trainer.evalbatches()
    return metric_dict["Accuracy"], elapsed


def count_params(model) -> int:
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

In [ ]:
def pixel_loaders(cfg, batch_size: int = 32):
    """The other collate_fn: raw images instead of cached vectors.

    Resizing and one-hotting happen here, per batch, instead of being written to disk — which is
    why the CNN path needs no cache and no preprocessing pass.
    """
    size, channels = cfg["size"], cfg["channels"]
    input_col, label_col = cfg["schema"].input_col, cfg["schema"].label_col
    num_classes = cfg["num_classes"]

    def process(item):
        img = item[input_col].convert("L" if channels == 1 else "RGB").resize(size)
        arr = np.asarray(img, dtype=np.float32) / 255.0
        if channels == 1:
            arr = arr[..., None]
        return torch.from_numpy(arr).permute(2, 0, 1), item[label_col]

    def collate(items):
        X, y = zip(*[process(i) for i in items])
        onehot = torch.nn.functional.one_hot(torch.tensor(y), num_classes).float()
        return torch.stack(X), onehot

    return (
        DataLoader(cfg["train"], batch_size=batch_size, shuffle=True, collate_fn=collate),
        DataLoader(cfg["test"], batch_size=batch_size, shuffle=False, collate_fn=collate),
    )


def make_cnn(cfg, filters: int = 128):
    return CNN(
        filters=filters,
        units1=128,
        units2=128,
        input_size=(32, cfg["channels"], *cfg["size"]),
        num_classes=cfg["num_classes"],
    )

In [ ]:
for name, cfg in datasets.items():
    print(f"\n{'=' * 20} {name}: frozen encoder + head {'=' * 20}")
    trainloader, testloader = cached_loaders(cfg)
    head = make_head(cfg)
    cfg["head_acc"], cfg["head_time"] = train(head, trainloader, testloader)
    cfg["head_params"] = count_params(head)

    print(f"\n{'=' * 20} {name}: CNN from scratch {'=' * 20}")
    ptrain, ptest = pixel_loaders(cfg)
    cnn = make_cnn(cfg)
    cfg["cnn_acc"], cfg["cnn_time"] = train(cnn, ptrain, ptest)
    cfg["cnn_params"] = count_params(cnn)

## 3.7 The comparison

Same images, same labels, same trainer, same number of epochs. The only thing that differs is
where the representation came from.

In [ ]:
for name, cfg in datasets.items():
    n_images = len(cfg["train"]) + len(cfg["test"])
    print(f"\n{name}  ({cfg['num_classes']} classes, {len(cfg['train'])} train images)")
    print(f"  {'':26} {'CNN from scratch':>18} {'frozen + head':>18}")
    print(f"  {'accuracy':26} {cfg['cnn_acc']:>18.3f} {cfg['head_acc']:>18.3f}")
    print(f"  {'trainable parameters':26} {cfg['cnn_params']:>18,} {cfg['head_params']:>18,}")
    print(f"  {'training time':26} {cfg['cnn_time']:>17.0f}s {cfg['head_time']:>17.0f}s")
    print(f"  {'one-time encoding (paid)':26} {'--':>18} {cfg['embed_time']:>17.0f}s")
    print(f"  {'  ... at the measured rate':26} {'--':>18} {n_images / rate:>17.0f}s")

Read the numbers, then read this.

**Look at the parameter counts before you draw the obvious conclusion.** The head has *more*
trainable parameters than the CNN, not fewer — a `Linear` over a 1280-dimensional embedding is a
big matrix. And it still trains an order of magnitude faster. That is worth sitting with, because
"number of parameters" is the metric people reach for when they mean "cost", and here the two
point in opposite directions.

The reason is what each model does per example. The CNN slides filters over 64x64x3 pixels through
three convolution-and-pool stages, forwards and backwards, every epoch. The head multiplies one
precomputed 1280-vector by two matrices. The convolutions are where the compute is; the parameters
are just where the memory is.

**So the cost profile is the point, not the accuracy.** The head trains in seconds and could be
retrained twenty times this afternoon. The encoding pass is the expensive half — and it is paid
**once**, by whoever has the hardware, and then it is a folder anybody can download. If you had to
re-encode every epoch, none of this would be worth doing.

**Which column a hyperparameter search multiplies is the whole argument.** A search over the CNN
multiplies the training column, because every trial retrains the convolutions. A search over the
head multiplies the *head* column, and leaves the encoding cost where it is: paid once, shared
across every trial. That is why the next notebook can afford to search, and it is exercise E3.

**The gap is much bigger on flowers than on fashion.** On fashion the two are close — a few points
apart. On flowers they are not remotely close. That is not an accident, and it is where the
honesty lives. Fashion MNIST is 28x28 greyscale clothing icons: ImageNet features are a poor fit
for them, the images have to be upscaled to 224x224 to be encoded at all, and a small CNN is
already near its ceiling. Flowers is 102 fine-grained classes of natural photographs with about 39
examples each, which is exactly what a pretrained encoder is for — it has already seen petals and
textures and stems, and your CNN has 39 pictures per class to discover them from.

**Be careful how you state the flowers result.** Look at the per-epoch log rather than the final
number: the CNN's loss falls steadily and its accuracy is still climbing when the ten epochs run
out. It is not broken and it is not stuck — it is *far from converged*, and it would need many
more epochs, real augmentation, and probably normalisation layers to close much of that gap. What
the comparison actually shows is not "a CNN cannot do this". It is:

> for the same budget — same data, same epochs, same trainer — the frozen encoder is already
> finished and the from-scratch model has barely started.

Budget is the whole comparison. State it that way, and the result is honest; state it as "CNNs are
bad at flowers" and you have learned the wrong thing.

So the rule is not "pretrained always wins". It is:

> a frozen encoder wins where its pretraining distribution matches your data and your data is
> scarce — and it has least to offer where your task looks nothing like what it was trained on
> and you have plenty of labels.

Fashion is the second case. Say that out loud, because a course that only ever showed the first
one would be selling something.

## 3.8 Look back at lesson 1

Open `1_pytorch_intro/07_vectors.ipynb` next to this notebook and compare the pipelines:

| | IMDB reviews (L1) | flowers (here) |
|---|---|---|
| vectorizer | `Vectorizer` (a text transformer) | `ImageVectorizer` (a vision model) |
| item shape | `(C, D)` — chunked, varying `C` | `(D)` — one vector |
| padder | `FixedPadding(max_chunks=2)` | `torch.stack` |
| aggregator | `MaskedMeanAggregator` | none needed |
| collate | `Collate` | `Collate` |
| head | `Serial([... NeuralNet])` | `Serial([... NeuralNet])` |
| trainer | `mltrainer` | `mltrainer` |

Two modalities, nothing in common at the input, and the bottom three rows are identical code.
That is what "working at the vector level" buys: once something is a vector, the pipeline stops
caring where it came from. In lesson 5 you will fuse two of these streams into one model, and this
is why that is a three-line change instead of a rewrite.

## Exercises

**E1 — swap the encoder.** Change `model_name` to `facebook/dinov2-small` (22M parameters, 384
dimensions) and re-run. Record accuracy *and* the wall-clock of the encoding pass for each. Then
plot accuracy against encoding cost for `mobilenet_v2`, `microsoft/resnet-18` and `dinov2-small`.

Two things to notice. First, a bigger encoder costs nothing at *training* time — the head is the
same size regardless, because only `hidden_size` changes. Second, "bigger" and "slower" are not
the same axis: measure both, and check whether the ranking is what you expected.

**E2 — where does the noise stop helping?** Try `GaussianNoise(sigma=)` at `0.0`, `0.05`, `0.2`,
`0.5` on flowers. Then shrink the training set to 500 images and repeat. Form a hypothesis about
the relationship between training-set size and the useful amount of noise *before* you run the
second sweep.

**E2b — give the CNN a fair fight.** The flowers comparison gave both models ten epochs, and the
CNN was still improving when it ran out. Find out what it would actually take: raise `EPOCHS` for
the CNN only, and add `nn.BatchNorm2d` after the convolutions (you will meet batchnorm properly in
the lesson slides). How many epochs before it reaches the head's accuracy — and how does the
wall-clock of getting there compare with the one-time encoding cost? If it never gets there, say
so, and say what you would try next.

**E3 — when does embedding once pay for itself?** (This one sets up `03_mlflow.ipynb`.) You are
going to run a hyperparameter search with `n` trials. Write down the total cost of each approach
as a function of `n`, using the numbers you measured in §3.7:

- searching over the CNN: `n * cnn_time`
- searching over the head: `encoding_time + n * head_time`

Solve for the `n` where they cross. Then check your arithmetic against the two searches in the
next notebook. If the crossover is at `n = 2`, what does that tell you about how the rest of this
course is going to be structured?